<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# Guia de Instalacao: ORB-SLAM 3

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

## Visao Geral

Este guia reune todos os comandos e configuracoes necessarias para compilar e executar o **ORB-SLAM 3** do zero.

### Cadeia de dependencias

```
Ubuntu 18.04 (base)
  └─ Dependencias do sistema (build-essential, cmake, libeigen3-dev, ...)
      └─ OpenCV 3.2.0 (com patch FFmpeg)
      └─ Pangolin (commit especifico)
          └─ ORB-SLAM 3 (commit especifico + patch LoopClosing.h)
```

### Versoes testadas

| Componente | Versao |
|---|---|
| Ubuntu | 18.04 LTS |
| OpenCV | 3.2.0 |
| Pangolin | commit `86eb4975fc4fc8b5d92148c2e370045ae9bf9f5d` |
| ORB-SLAM 3 | commit `ef9784101fbd28506b52f233315541ef8ba7af57` |
| CMake | >= 3.10 |

## Pre-requisitos

- **Ubuntu 18.04** recem-instalado ([download da imagem ISO](https://releases.ubuntu.com/18.04/))
- Pode ser maquina virtual (VirtualBox, UTM) ou bare-metal
- Conexao com internet para baixar pacotes e repositorios
- Minimo 4 GB de RAM e 20 GB de disco

> **Aviso:** Outras versoes do Ubuntu (20.04, 22.04) podem nao funcionar sem adaptacoes significativas. Este guia foi testado exclusivamente no Ubuntu 18.04.

## Passo 1: Dependencias do Sistema

Adicionar o repositorio de seguranca Xenial e instalar todas as bibliotecas necessarias.

```bash
sudo add-apt-repository "deb http://security.ubuntu.com/ubuntu xenial-security main"
sudo apt update
```

```bash
sudo apt-get install -y \
    build-essential \
    cmake \
    git \
    libgtk2.0-dev \
    pkg-config \
    libavcodec-dev \
    libavformat-dev \
    libswscale-dev \
    python-dev \
    python-numpy \
    libtbb2 \
    libtbb-dev \
    libjpeg-dev \
    libpng-dev \
    libtiff-dev \
    libdc1394-22-dev \
    libjasper-dev \
    libglew-dev \
    libboost-all-dev \
    libssl-dev \
    libeigen3-dev
```

## Passo 2: OpenCV 3.2.0 (com patch FFmpeg)

### Clonar e selecionar a versao

```bash
cd ~
mkdir Dev && cd Dev
git clone https://github.com/opencv/opencv.git
cd opencv
git checkout 3.2.0
```

### Patch obrigatorio no FFmpeg

Abrir o arquivo `./modules/videoio/src/cap_ffmpeg_impl.hpp` e adicionar as seguintes linhas **no inicio do arquivo** (antes dos includes existentes):

```cpp
#define AV_CODEC_FLAG_GLOBAL_HEADER (1 << 22)
#define CODEC_FLAG_GLOBAL_HEADER AV_CODEC_FLAG_GLOBAL_HEADER
#define AVFMT_RAWPICTURE 0x0020
```

Sem esse patch, a compilacao falha com erros relacionados a constantes FFmpeg depreciadas.

### Compilar e instalar

```bash
mkdir build && cd build
cmake -D CMAKE_BUILD_TYPE=Release \
      -D WITH_CUDA=OFF \
      -D CMAKE_INSTALL_PREFIX=/usr/local ..
make -j $(nproc)
sudo make install
```

## Passo 3: Pangolin

Pangolin e a biblioteca de visualizacao 3D usada pelo ORB-SLAM 3.

```bash
cd ~/Dev
git clone https://github.com/stevenlovegrove/Pangolin.git
cd Pangolin
git checkout 86eb4975fc4fc8b5d92148c2e370045ae9bf9f5d
```

```bash
mkdir build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release
make -j $(nproc)
sudo make install
```

## Passo 4: ORB-SLAM 3

### Clonar e selecionar o commit

```bash
cd ~/Dev
git clone https://github.com/UZ-SLAMLab/ORB_SLAM3.git
cd ORB_SLAM3
git checkout ef9784101fbd28506b52f233315541ef8ba7af57
```

### Patch obrigatorio no LoopClosing.h

Abrir `./include/LoopClosing.h` e na **linha 51**, alterar:

**De:**
```cpp
Eigen::aligned_allocator<std::pair<const KeyFrame*, g2o::Sim3> > > KeyFrameAndPose;
```

**Para:**
```cpp
Eigen::aligned_allocator<std::pair<KeyFrame *const, g2o::Sim3> > > KeyFrameAndPose;
```

A diferenca e sutil: `const KeyFrame*` vira `KeyFrame *const`. Sem essa mudanca a compilacao falha.

### Compilar

```bash
./build.sh
```

> **Dica:** Se falhar na primeira tentativa, execute `./build.sh` mais uma ou duas vezes. Em alguns ambientes a compilacao so funciona na segunda execucao.

## Passo 5: Testar com o Dataset EuRoc

### Baixar o dataset

```bash
cd ~
mkdir -p Datasets/EuRoc
cd Datasets/EuRoc/
wget -c http://robotics.ethz.ch/~asl-datasets/ijrr_euroc_mav_dataset/machine_hall/MH_01_easy/MH_01_easy.zip
mkdir MH01
unzip MH_01_easy.zip -d MH01/
```

### Executar o ORB-SLAM 3 (monocular)

```bash
cd ~/Dev/ORB_SLAM3
./Examples/Monocular/mono_euroc \
    ./Vocabulary/ORBvoc.txt \
    ./Examples/Monocular/EuRoC.yaml \
    ~/Datasets/EuRoc/MH01 \
    ./
```

Voce devera ver duas janelas:
1. **Sequencia de imagens** com rastreadores verdes sobre as features detectadas
2. **Nuvem de pontos 3D** com a trajetoria estimada da camera

## Troubleshooting

### Erro de compilacao no OpenCV: constantes FFmpeg
**Sintoma:** `error: 'AV_CODEC_FLAG_GLOBAL_HEADER' was not declared`

**Solucao:** Aplicar o patch no `cap_ffmpeg_impl.hpp` (Passo 2).

---

### Erro de compilacao no ORB-SLAM 3: LoopClosing.h
**Sintoma:** `error: no matching function for call to 'std::pair<...>'`

**Solucao:** Aplicar o patch no `LoopClosing.h` (Passo 4).

---

### build.sh falha na primeira execucao
**Sintoma:** Erros de linking ou dependencias nao encontradas.

**Solucao:** Executar `./build.sh` novamente (ate 2-3 vezes).

---

### Pangolin nao encontrado
**Sintoma:** `Could not find Pangolin` ao compilar ORB-SLAM 3.

**Solucao:** Confirmar que `sudo make install` do Pangolin foi executado. Verificar se `/usr/local/lib/libpangolin.so` existe.

---

### Erro com Eigen
**Sintoma:** Headers do Eigen nao encontrados.

**Solucao:** `sudo apt-get install libeigen3-dev` e verificar se `/usr/include/eigen3` existe.

---

### Ubuntu 20.04 ou mais recente
**Sintoma:** Varios erros de compilacao com dependencias mais novas.

**Solucao:** Usar Ubuntu 18.04. As versoes mais recentes mudam versoes de bibliotecas (GCC, Boost, OpenCV) que sao incompativeis com este commit do ORB-SLAM 3.